In [ ]:
""" 
A Positional Encoder (PE) is necessary to learn concepts and operations conditioned on the
relative distance of tokens in the sequence instead of absolute positions. If we had a PE with absolute
positions, a sequence like "3 + 4 = 7" that was seen often early in the sequence could not be solved 
if it appeared late in the sequence during inference.

The Rotary Positional Encoding (RoPE) integrates the PE into the attention formulation instead of concatenating it to the token embeddings,
which could distorts less the meaning of the embeddings. Additionally, it natively results in decaying inter-token depency with
increasing relative distance and can be applied to linear self-attention. 
ReFormer paper: https://arxiv.org/abs/2104.09864 

"""

In [22]:
import math
import torch


def precompute_freqs_cis() -> torch.Tensor:
    dim = 1
    seq_len = 10
    base = 10000

    # Frequency calculation for thetas (look ReFormer page 5)
    thetas = 1 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32)/dim))

    # All possible positions (bound by seq_le)
    t = torch.arange(seq_len)

    # Outer product: [positions] x [thetas] -> m*theta
    freqs = torch.outer(t, thetas)

    # Convert to complex exponentials: for ang=freq*pos:  e^(i*ang) = cos(ang) * i*sin(ang)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # z=R*e^(i*theta); Radius = 1, theta=ang
    return freqs_cis        # [seq_len, d/2]


def apply_rot_emb(x: torch.Tensor, freq_cis: torch.Tensor) -> torch.Tensor:
    # x: [B, S, H, D] => [B, S, H, D/2, 2]
    # freqs: [S, D/2]
    dtype = x.dtype
    x = torch.view_as_complex(x.float().view(*x.shape[:-1], -1, 2)) # [B, S, H, D/2]
    freq_cis = freq_cis.view(1, x.size(1), 1, x.size(-1)) # [1, S, 1, D/2]
    y = torch.view_as_real(x*freq_cis).flatten(3)
    return y.to(dtype)


x = torch.ones((1,10,1,2))
freqs = precompute_freqs_cis()
x_rope = apply_rot_emb(x, freqs)
print("from ones to: ", x_rope.flatten()[:, None])

from ones to:  tensor([[ 1.0000],
        [ 1.0000],
        [-0.3012],
        [ 1.3818],
        [-1.3254],
        [ 0.4932],
        [-1.1311],
        [-0.8489],
        [ 0.1032],
        [-1.4104],
        [ 1.2426],
        [-0.6753],
        [ 1.2396],
        [ 0.6808],
        [ 0.0969],
        [ 1.4109],
        [-1.1349],
        [ 0.8439],
        [-1.3232],
        [-0.4990]])
